<a href="https://colab.research.google.com/github/smunazza/231A046_AIDS_Sem7_NLP_Experiments/blob/main/nlp_exp5.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [24]:
import nltk
from nltk import word_tokenize, bigrams, FreqDist, ConditionalFreqDist
from nltk.corpus import stopwords
from collections import defaultdict

In [25]:
nltk.download('punkt')
nltk.download('stopwords')
nltk.download('punkt_tab')

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True

In [26]:
def preprocess(text):
  text=text.lower
  tokens=word_tokenize(text)
  stop_words=set(stopwords.words('english'))
  tokens=[token for token in tokens if token.isalnum() and token not in stop_words]
  return tokens

In [27]:
corpus="the sun is shining, the sky is blue, the sun is bright and the sky is dark"

def preprocess(text):
  text=text.lower()
  tokens=word_tokenize(text)
  stop_words=set(stopwords.words('english'))
  tokens=[token for token in tokens if token.isalnum() and token not in stop_words]
  return tokens

tokens=preprocess(corpus)
print(tokens)

['sun', 'shining', 'sky', 'blue', 'sun', 'bright', 'sky', 'dark']


In [28]:
bigrams_list=list(bigrams(tokens))
print(bigrams_list)

[('sun', 'shining'), ('shining', 'sky'), ('sky', 'blue'), ('blue', 'sun'), ('sun', 'bright'), ('bright', 'sky'), ('sky', 'dark')]


In [29]:
bigram_freq=FreqDist(bigrams_list)
bigram_freq

FreqDist({('sun', 'shining'): 1, ('shining', 'sky'): 1, ('sky', 'blue'): 1, ('blue', 'sun'): 1, ('sun', 'bright'): 1, ('bright', 'sky'): 1, ('sky', 'dark'): 1})

In [30]:
cfd=ConditionalFreqDist(bigrams_list)
print(cfd.items())

dict_items([('sun', FreqDist({'shining': 1, 'bright': 1})), ('shining', FreqDist({'sky': 1})), ('sky', FreqDist({'blue': 1, 'dark': 1})), ('blue', FreqDist({'sun': 1})), ('bright', FreqDist({'sky': 1}))])


In [31]:
from nltk import trigrams
trigrams_list=list(trigrams(tokens))
print(trigrams_list)

[('sun', 'shining', 'sky'), ('shining', 'sky', 'blue'), ('sky', 'blue', 'sun'), ('blue', 'sun', 'bright'), ('sun', 'bright', 'sky'), ('bright', 'sky', 'dark')]


In [32]:
def predict_next_word(word):
  word=word.lower()
  if word in cfd:
    next_word=cfd[word].max()
    return next_word
  else:
    return None

In [33]:
start_word="sky"
predicted_word=predict_next_word(start_word)
if predicted_word:
  print(f"The predicted word after '{start_word}' is '{predicted_word}'")
else:
  print(f"No prediction found for '{start_word}'")
print(predicted_word)

The predicted word after 'sky' is 'blue'
blue


In [34]:
def generate_sequence(start_word, length=8):
  sequence=[start_word]
  current_word=start_word
  for _ in range(length):
    next_word=predict_next_word(current_word)
    if next_word:
      sequence.append(next_word)
      current_word=next_word
    else:
      break
  return ' '.join(sequence)


generated_sequence=generate_sequence("shining", length=8)
print(f"generated sequence is: {generated_sequence}")


generated sequence is: shining sky blue sun shining sky blue sun shining


PART B: USING TRAINING DATA OF REUTERS

In [4]:
import nltk
from nltk import bigrams, trigrams, ngrams
from nltk.corpus import reuters
from collections import defaultdict

In [5]:
nltk.download('reuters')
nltk.download('punkt_tab')

[nltk_data] Downloading package reuters to /root/nltk_data...
[nltk_data]   Package reuters is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True

In [6]:
file_ids=reuters.fileids()
print(file_ids[0:100])

['test/14826', 'test/14828', 'test/14829', 'test/14832', 'test/14833', 'test/14839', 'test/14840', 'test/14841', 'test/14842', 'test/14843', 'test/14844', 'test/14849', 'test/14852', 'test/14854', 'test/14858', 'test/14859', 'test/14860', 'test/14861', 'test/14862', 'test/14863', 'test/14865', 'test/14867', 'test/14872', 'test/14873', 'test/14875', 'test/14876', 'test/14877', 'test/14881', 'test/14882', 'test/14885', 'test/14886', 'test/14888', 'test/14890', 'test/14891', 'test/14892', 'test/14899', 'test/14900', 'test/14903', 'test/14904', 'test/14907', 'test/14909', 'test/14911', 'test/14912', 'test/14913', 'test/14918', 'test/14919', 'test/14921', 'test/14922', 'test/14923', 'test/14926', 'test/14928', 'test/14930', 'test/14931', 'test/14932', 'test/14933', 'test/14934', 'test/14941', 'test/14943', 'test/14949', 'test/14951', 'test/14954', 'test/14957', 'test/14958', 'test/14959', 'test/14960', 'test/14962', 'test/14963', 'test/14964', 'test/14965', 'test/14967', 'test/14968', 'test

In [7]:
len(file_ids)

10788

In [8]:
print(reuters.words()[:60])

['ASIAN', 'EXPORTERS', 'FEAR', 'DAMAGE', 'FROM', 'U', '.', 'S', '.-', 'JAPAN', 'RIFT', 'Mounting', 'trade', 'friction', 'between', 'the', 'U', '.', 'S', '.', 'And', 'Japan', 'has', 'raised', 'fears', 'among', 'many', 'of', 'Asia', "'", 's', 'exporting', 'nations', 'that', 'the', 'row', 'could', 'inflict', 'far', '-', 'reaching', 'economic', 'damage', ',', 'businessmen', 'and', 'officials', 'said', '.', 'They', 'told', 'Reuter', 'correspondents', 'in', 'Asian', 'capitals', 'a', 'U', '.', 'S']


In [9]:
word1=reuters.words(file_ids[10])
len(word1)

116

In [10]:
words=reuters.words(file_ids[10787])
print(words[:100])

['&', 'lt', ';', 'A', '.', 'H', '.', 'A', '.', 'AUTOMOTIVE', 'TECHNOLOGIES', 'CORP', '>', 'YEAR', 'NET', 'Shr', '43', 'cts', 'vs', '52', 'cts', 'Shr', 'diluted', '41', 'cts', 'vs', '49', 'cts', 'Net', '1', ',', '916', ',', '000', 'vs', '2', ',', '281', ',', '000', 'Revs', '32', '.', '6', 'mln', 'vs', '22', '.', '6', 'mln']


In [11]:
words=nltk.word_tokenize(" ".join(reuters.words()))


In [12]:
len(words)

1728932

In [13]:
tri_grams=list(trigrams(words))
print(tri_grams[:10])

[('ASIAN', 'EXPORTERS', 'FEAR'), ('EXPORTERS', 'FEAR', 'DAMAGE'), ('FEAR', 'DAMAGE', 'FROM'), ('DAMAGE', 'FROM', 'U'), ('FROM', 'U', '.'), ('U', '.', 'S'), ('.', 'S', '.-'), ('S', '.-', 'JAPAN'), ('.-', 'JAPAN', 'RIFT'), ('JAPAN', 'RIFT', 'Mounting')]


In [14]:
model = defaultdict(lambda: defaultdict(lambda: 0))

In [15]:
for w1, w2, w3 in tri_grams:
 model[(w1,w2)][w3]+=1

In [20]:
for w1,w2 in model:
  total_count=float(sum(model[w1,w2].values()))
  for w3 in model[w1,w2]:
    model[w1,w2][w3]/=total_count


In [37]:
def predict_next_word(w1, w2):
  next_word=model[w1,w2]
  if next_word:
    predicted_word=max(next_word, key=next_word.get)
    return predicted_word
  else:
    return 'no prediction available'

In [39]:
print("next word:", predict_next_word("FROM", "U"))

next word: .


In [46]:
def calculate_perplexity(test_text, ngram_probs):

  n=len(words)
  log_prob_sum=0

  for i in range(len(tokens)-1):
    w1,w2=tokens[i],tokens[i+1]
    prob=model[w1][w2] if w2 in model[w1] else 1e-6
    log_prob_sum+=math.log(prob)

    perplexity= math.exp(-log_prob_sum/(n-1))
    return perplexity

In [53]:
import math
from nltk import word_tokenize
words="the damage is from japan"
tokens=word_tokenize(words)

In [54]:
calculate_perplexity(words, model)

1.823348000868441